In [1]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(5)

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-05-08,3.71,3.69,3.74,3.75,3.90,3.92,4.02,4.19,4.38,4.93,4.95
2026-05-11,3.71,3.70,3.77,3.79,3.95,3.96,4.07,4.24,4.42,4.97,4.98
2026-05-12,3.71,3.70,3.77,3.80,4.00,4.01,4.12,4.29,4.46,5.02,5.03
2026-05-13,3.71,3.69,3.77,3.79,3.98,4.00,4.12,4.28,4.46,5.03,5.03
2026-05-14,3.72,3.69,3.76,3.79,4.00,4.04,4.13,4.29,4.47,5.01,5.02


In [3]:
#split data int training and testing sets

train_df = df[df.index <= "2025-4-17"].reset_index(drop=True)
test_df = df[df.index > "2025-4-17"]
matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]


In [4]:
lagtest = []
lag_order = []

model = VAR(train_df)
for i in range(1,31):
    result = model.fit(i)
    lagtest.append(result)

    #k_ar is the number of lagged observations (just lag count basically)
    lag_order.append(result.k_ar)



forecastArr = []
for i in range(30):
    #create a forecast for each lag
    #forecast 20 days in the future
    forecastArr.append(lagtest[i].forecast(
        train_df.values[-(lag_order[i]):],
        steps=20
    ))

errors = []

for i in range(30):
    error1day = forecastArr[i][0] - test_df.iloc[0].to_numpy()
    error5day = forecastArr[i][4] - test_df.iloc[4].to_numpy()
    error20day = forecastArr[i][19] - test_df.iloc[19].to_numpy()
    
    errors.append([error1day, error5day, error20day])

rmse = []
for i in range(30):
    rmse1day = np.sqrt(np.mean(errors[i][0]**2))
    rmse5day = np.sqrt(np.mean(errors[i][1]**2))
    rmse20day = np.sqrt(np.mean(errors[i][2]**2))

    rmse.append([rmse1day, rmse5day, rmse20day])


horizons = [1,5,20]

'''
for i in range(len(rmse)):
    print(f"Lag: {lag_order[i]} \nRMSE for 1 Day forecast{rmse[i][0]} \nRMSE for 5 Day forecast: {rmse[i][1]} \nRMSE for 20 Day forecast: {rmse[i][2]} \n\n")
'''

aggregatedError = {}
for i in range(len(rmse)):
    aggregatedError[lag_order[i]] = sum(rmse[i])
     
for key,value in sorted(aggregatedError.items(), key = lambda item: item[1]):
    print(key, " : ", value)

23  :  0.23411100050342942
21  :  0.2423754960935092
22  :  0.2441051874364819
24  :  0.2450362465110384
19  :  0.24661508334522741
10  :  0.24702327195889964
20  :  0.24716863400422812
11  :  0.24771098187480578
9  :  0.24822346249183588
18  :  0.2488769053542329
15  :  0.2497096184843071
12  :  0.25002821637598993
14  :  0.25170255817124987
16  :  0.2522391312099287
8  :  0.25230328923674106
13  :  0.25364130492081305
25  :  0.2547875699514152
17  :  0.2554483232977176
7  :  0.2607121022577713
6  :  0.2638067563495348
27  :  0.2679445763789831
5  :  0.2681219655761373
26  :  0.2705581736827214
28  :  0.2718209283272123
4  :  0.2748944340290923
1  :  0.27660920698571817
3  :  0.27855726603570025
29  :  0.2806814051726296
30  :  0.28490510852130113
2  :  0.2865071698532189


In [5]:
#rolling window forecast for 1, 5, and 20 day horizons
horizons = [1, 5, 20]

#indexes for windows
windowStarts = range(len(df) - 41, len(df) - 20)

def rollingMetrics(errors, predictedChanges, actualChanges):
    rmse = 100 * np.sqrt(np.mean(np.array(errors)**2))
    mae = 100 * np.mean(np.abs(errors))
    directionalAccuracy = 100 * np.mean(np.sign(predictedChanges) == np.sign(actualChanges))
    return rmse, mae, directionalAccuracy

#store all metrics
metrics = [[[] for _ in range(3)] for _ in range(len(matList))]

#aggregate metrics to analyze error per horizon
sum = [[0.0,0.0,0.0] for _ in range(3)]

#use a VAR model with 5 lags
lag = 5

for i in range(len(matList)):
    
    errors = [[], [], []]
    predictedChanges = [[], [], []]
    actualChanges = [[], [], []]
    #first array is horizon, second array is the rolling window number

    for start in windowStarts:
        train = df.iloc[:start + 1].reset_index(drop=True)
        result = VAR(train).fit(lag)
        forecast = result.forecast(train.values[-lag:], steps=20)

        for j in range(len(horizons)):
            horizon = horizons[j]
            predicted = forecast[horizon - 1][i]
            actual = df[matList[i]].iloc[start + horizon]
            current = df[matList[i]].iloc[start]

            #predicted changes and actual changes are for directional accuracy
            errors[j].append(predicted - actual)
            predictedChanges[j].append(predicted - current)
            actualChanges[j].append(actual - current)

    print("Lag: ", lag, " Maturity: ", matList[i])
    for j in range(len(horizons)):
        rmse, mae, directionalAccuracy = rollingMetrics(errors[j], predictedChanges[j], actualChanges[j])
        metrics[i][j].append([rmse, mae, directionalAccuracy])
        sum[j][0]+=rmse**2
        sum[j][1]+=mae
        sum[j][2]+=directionalAccuracy
        print("Forecast ", horizons[j], " day, RMSE: ", rmse, " MAE: ", mae, " Directional Accuracy: ", directionalAccuracy, "%")
    print("\n")

print("Lag: ", lag)
for i in range(len(sum)):
    for j in range(3):
        sum[i][j]= sum[i][j]/len(matList)
    sum[i][0] = np.sqrt(sum[i][0])
    print(f"Horizon: {horizons[i]} day, rmse: {sum[i][0]}, MAE: {sum[i][1]}, Directional Accuracy: {sum[i][2]} \n")

Lag:  5  Maturity:  0Y1M
Forecast  1  day, RMSE:  1.4248974059349933  MAE:  1.1064080625841108  Directional Accuracy:  42.857142857142854 %
Forecast  5  day, RMSE:  2.4118763762415063  MAE:  1.8662534269942719  Directional Accuracy:  80.95238095238095 %
Forecast  20  day, RMSE:  3.445773887778832  MAE:  2.6912614957212893  Directional Accuracy:  61.904761904761905 %


Lag:  5  Maturity:  0Y3M
Forecast  1  day, RMSE:  1.1805132589394431  MAE:  0.9229294725149149  Directional Accuracy:  33.33333333333333 %
Forecast  5  day, RMSE:  3.161456592717421  MAE:  2.7354829017182296  Directional Accuracy:  23.809523809523807 %
Forecast  20  day, RMSE:  7.586192094183536  MAE:  7.31606168932373  Directional Accuracy:  4.761904761904762 %


Lag:  5  Maturity:  0Y6M
Forecast  1  day, RMSE:  1.7771276387328352  MAE:  1.4231760565909024  Directional Accuracy:  38.095238095238095 %
Forecast  5  day, RMSE:  4.116389281399956  MAE:  3.244223331995645  Directional Accuracy:  19.047619047619047 %
Forecast 